**Nawaf Khalid Alsabr**  
**ID: 2230004159**

# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [19]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [20]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

**Answers:**

| # | Task | Best-suited model family | Why |
|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** (e.g. BERT, DistilBERT) | The whole input needs to be read and understood at once to output a single label — no generation is required. |
| 2 | Machine translation | **Encoder–decoder** (e.g. T5, BART) | The encoder reads and understands the source sentence, and the decoder generates the translated sentence — a classic sequence-to-sequence task. |
| 3 | Open-ended text generation | **Decoder-only** (e.g. GPT family) | The model only needs to predict the next token given everything so far, with no fixed-length input to "understand" first. |
| 4 | Named Entity Recognition | **Encoder-only** (e.g. BERT, RoBERTa) | NER is a classification task per token — the model needs a rich understanding of the full input, not text generation. |
| 5 | Text summarization | **Encoder–decoder** (e.g. T5, BART) | The encoder reads and understands the full document, and the decoder generates a shorter version of it — another sequence-to-sequence task. |

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [ ]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"Sentence: {sentence!r}")
    print(f"  -> Label: {result['label']}, Score: {result['score']:.4f}")


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [ ]:
paragraph = (
    "Natural language processing (NLP) is a field of artificial intelligence that focuses on "
    "the interaction between computers and human language. It combines computational "
    "linguistics with machine learning and deep learning to enable computers to understand, "
    "interpret, and generate human language. NLP is used in many everyday applications, "
    "including search engines, virtual assistants, translation services, and spam filters."
)

summarize_prompt = "Summarize: " + paragraph

summary_output = model.generate(**tokenizer(summarize_prompt, return_tensors="pt"), max_new_tokens=60)

print("Original paragraph:")
print(paragraph)
print("\nSummary:")
print(tokenizer.decode(summary_output[0], skip_special_tokens=True))


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A chatbot's core job is to keep generating the next most plausible word given everything said so far in the conversation — exactly what a decoder-only model is trained to do, since it predicts each token autoregressively based only on the preceding tokens. This also makes it naturally suited to open-ended, variable-length output: a chatbot reply isn't a fixed-size label or a bounded translation, it's free-form text that can be as short or long as needed, which decoder-only generation handles well. Decoder-only models are also straightforward to use conversationally — the entire chat history (system prompt, previous turns, and the latest user message) can simply be fed in as one continuous sequence of preceding tokens, with no separate encoding step required. Finally, because decoder-only architectures like GPT are trained on massive amounts of general text to predict "what comes next," they pick up broad world knowledge and conversational patterns that transfer well to open dialogue, unlike encoder-only models which are built for understanding/classifying fixed input rather than producing new text.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.
